<a href="https://colab.research.google.com/github/Shaifali07/langgraph_learning/blob/main/UPSC_essay_evalution_langgraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Annotated
from operator import add

In [ ]:
!pip install langchain-groq

In [ ]:
from google.colab import userdata
import os

groq_api_key = userdata.get('GROQ_API_KEY')
os.environ["GROQ_API_KEY"] = groq_api_key

In [ ]:
!pip install langchain_core

In [ ]:
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
class ExampleModelCoT(BaseModel):
    Clarity_of_Thoughts: str = Field(description="Feedback on how clear the essay is in thoughts")
    Clarity_of_Thoughts_score:float = Field(description="Score out of 10 given to essay by assesing it in clarity of thoughts")
class ExampleModelDoC(BaseModel):
    Depth_of_Concept: str = Field(description="Feedback on how much deep the context and concept has been covered")
    Depth_of_Thoughts_score:float = Field(description="Score out of 10 given to essay by assesing it in depth of concept")
class ExampleModellanguage(BaseModel):
    language: str = Field(description="Feedback on how correct the grammar and language is")
    language_score:float = Field(description="Score out of 10 given to essay by asssesing it in language")
class ExampleModelsummary(BaseModel):
    summary: str = Field(description="Summary of all the feedbacks")


In [ ]:
class UPSC_essay_evaluation(TypedDict):
  topic:str
  Essay:str
  Clarity_of_Thoughts:str
  # Clarity_of_Thoughts_score:float
  Language:str
  Depth_of_Concept:str
  final_feedback:str
  individual_score:Annotated[list[float], add]
  final_score:float

In [ ]:
def caculate_CoT(s:UPSC_essay_evaluation):
  topic=s["topic"]
  essay=s["Essay"]
  parser = JsonOutputParser(pydantic_object=ExampleModelCoT)
  format_instructions = parser.get_format_instructions()
  prompt = ChatPromptTemplate.from_messages([
    ("system", "You  are UPSC exam evaluator. Evaluate the essay in terms of clarrity of the thoughts.The topic is {topic} and the essay by the candidate is {essay}. Generate feedback and score it. {format_instructions}" )])
  # prompt=f"You  are UPSC exam evaluator. Evaluate the essay in terms of clarrity of the thoughts.The topic is {topic} and the essay by the candidate is {essay}. Generate feedback and score it. {format_instructions} "
  llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0.5)
  chain = prompt | llm |parser
  result = chain.invoke({"topic": topic, "essay": essay,"format_instructions":format_instructions})
  return {'Clarity_of_Thoughts':result['Clarity_of_Thoughts'],'individual_score':[result['Clarity_of_Thoughts_score']]}

In [ ]:
def caculate_DoC(s:UPSC_essay_evaluation):
  topic=s["topic"]
  essay=s["Essay"]
  parser = JsonOutputParser(pydantic_object=ExampleModelDoC)
  format_instructions = parser.get_format_instructions()
  prompt = ChatPromptTemplate.from_messages([
    ("system", "You  are UPSC exam evaluator. Evaluate the essay in terms of depth of the concept covered.The topic is {topic} and the essay by the candidate is {essay}. Generate feedback and score it. {format_instructions}" )])
  # prompt=f"You  are UPSC exam evaluator. Evaluate the essay in terms of clarrity of the thoughts.The topic is {topic} and the essay by the candidate is {essay}. Generate feedback and score it. {format_instructions} "
  llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0.5)
  chain = prompt | llm |parser
  result = chain.invoke({"topic": topic, "essay": essay,"format_instructions":format_instructions})
  return {'Depth_of_Concept':result['language'],'individual_score':[result['Depth_of_Thoughts_score']]}

In [ ]:
def caculate_language(s:UPSC_essay_evaluation):
  topic=s["topic"]
  essay=s["Essay"]
  parser = JsonOutputParser(pydantic_object=ExampleModellanguage)
  format_instructions = parser.get_format_instructions()
  prompt = ChatPromptTemplate.from_messages([
    ("system", "You  are UPSC exam evaluator. Evaluate the essay in terms of language and grammar.The topic is {topic} and the essay by the candidate is {essay}. Generate feedback and score it. {format_instructions}" )])
  # prompt=f"You  are UPSC exam evaluator. Evaluate the essay in terms of clarrity of the thoughts.The topic is {topic} and the essay by the candidate is {essay}. Generate feedback and score it. {format_instructions} "
  llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0.5)
  chain = prompt | llm |parser
  result = chain.invoke({"topic": topic, "essay": essay,"format_instructions":format_instructions})
  return {'Language':result['language'],'individual_score':[result['language_score']]}

In [61]:
def final_score(s:UPSC_essay_evaluation):
  clarity_of_thoughts=s['Clarity_of_Thoughts']
  depth_of_concept=s['Depth_of_Concept']
  language=s['Language']
  parser = JsonOutputParser(pydantic_object=ExampleModelsummary)
  format_instructions = parser.get_format_instructions()
  prompt = ChatPromptTemplate.from_messages([
    ("system", "You  are UPSC exam eassay evaluator. Generate summary feedback form the given feedbacks. You have feedback of clarity of thoughts of that eassay here {clarity_of_thoughts}. Feedback for depth of concept is {depth_of_concept}. Feedback for language of the eassy is {language} follow{format_instructions} " )])

  llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0.5)
  chain = prompt | llm |parser
  result = chain.invoke({"clarity_of_thoughts": clarity_of_thoughts, "depth_of_concept": depth_of_concept,"language":language, "format_instructions":format_instructions})
  final_feedback=result['summary']
  individual_score=s['individual_score']
  final_score=sum(individual_score)/len(individual_score)
  return {'final_feedback':final_feedback,'final_score':final_score}

In [62]:
graph=StateGraph(UPSC_essay_evaluation)
graph.add_node("caculate_CoT",caculate_CoT)
graph.add_node("caculate_DoC",caculate_DoC)
graph.add_node("caculate_language",caculate_language)
graph.add_node("final_score",final_score)
graph.add_edge(START,"caculate_language")
graph.add_edge("caculate_language",'final_score')
graph.add_edge(START,"caculate_CoT")
graph.add_edge("caculate_CoT",'final_score')
graph.add_edge(START,"caculate_DoC")
graph.add_edge("caculate_DoC",'final_score')
graph.add_edge('final_score',END)
workflow=graph.compile()

In [63]:
topic="The Importance of Reading Books"
essay="Reading books is one of the best habits a person can develop. Books help us gain knowledge, improve our vocabulary, and expand our imagination. When we read, we learn about different cultures, ideas, and experiences that we might never encounter in real life. Reading also improves concentration and thinking skills. In today’s digital world, where many people spend time on screens, books provide a healthy and meaningful way to relax and learn at the same time."
inital_state={'topic':topic, "Essay":essay}
final_state=workflow.invoke(inital_state)
print(final_state)

{'topic': 'The Importance of Reading Books', 'Essay': 'Reading books is one of the best habits a person can develop. Books help us gain knowledge, improve our vocabulary, and expand our imagination. When we read, we learn about different cultures, ideas, and experiences that we might never encounter in real life. Reading also improves concentration and thinking skills. In today’s digital world, where many people spend time on screens, books provide a healthy and meaningful way to relax and learn at the same time.', 'Clarity_of_Thoughts': 'The essay is clear and concise in its thoughts, effectively conveying the importance of reading books. The candidate has provided a straightforward and easy-to-understand argument, highlighting the benefits of reading, such as gaining knowledge, improving vocabulary, and expanding imagination. However, the essay lacks depth and supporting examples, which could have further strengthened the argument.', 'Language': 'The language used in the essay is sim